# 200. LLM 流式 API：事件顺序、终态、断连恢复怎样实现？

> **面试问题：如何设计 token/tool delta、finish/error、usage、sequence 与 replay，使 SSE/流式 LLM API 不漏字、不重复拼接、不把半成品当完成？**

## 先给结论

面试中不能只背术语；需要把数学坐标、消息状态、协议顺序或模板字节流变成可检验的状态机。下列代码只使用标准库和小数组，明确教学 oracle 与生产替换点；它们不等同于真实模型效果、网络可靠性或正式安全认证。

## 一手资料

- [HTML Living Standard: Server-Sent Events](https://w3c.github.io/eventsource/)
- [RFC 6202](https://www.rfc-editor.org/rfc/rfc6202.html)
- [MCP Tasks Release Candidate](https://blog.modelcontextprotocol.io/posts/2026-07-28-release-candidate/)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "explicit-assertions", "production": "versioned-and-observed"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "explicit-assertions"  # 执行本行的状态、计算或校验逻辑。
assert "observed" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：流式输出是带终态的有序事件日志

token delta 只是事件之一。可靠 API 还必须标识 request、单调 sequence、文本/工具参数增量、usage、finish reason 和错误。客户端渲染是这些事件的投影，不能把收到第一段文本就当请求已经成功。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class StreamEvent:  # 执行本行的状态、计算或校验逻辑。
    request_id: str  # 执行本行的状态、计算或校验逻辑。
    sequence: int  # 执行本行的状态、计算或校验逻辑。
    kind: str  # 执行本行的状态、计算或校验逻辑。
    data: str  # 执行本行的状态、计算或校验逻辑。
events = [StreamEvent("r-1", 1, "text.delta", "你"), StreamEvent("r-1", 2, "text.delta", "好"), StreamEvent("r-1", 3, "finish", "stop")]  # 执行本行的状态、计算或校验逻辑。
assert events[0].sequence == 1  # 执行本行的状态、计算或校验逻辑。
assert events[-1].kind == "finish"  # 执行本行的状态、计算或校验逻辑。
assert len(events) == 3  # 执行本行的状态、计算或校验逻辑。


## 2. 顺序校验：每个 request 的 sequence 必须连续

网络重连或代理重复推送时，客户端不能盲目拼接。这里要求 sequence 从一连续递增；生产协议可使用 SSE `id`、last-event-id、分区 offset 或 server-side replay buffer，但仍要验证租户和请求归属。


In [ ]:
def validate_sequence(events):  # 执行本行的状态、计算或校验逻辑。
    if not events:  # 执行本行的状态、计算或校验逻辑。
        return False  # 执行本行的状态、计算或校验逻辑。
    request_ids = {event.request_id for event in events}  # 执行本行的状态、计算或校验逻辑。
    expected = list(range(1, len(events) + 1))  # 执行本行的状态、计算或校验逻辑。
    return len(request_ids) == 1 and [event.sequence for event in events] == expected  # 执行本行的状态、计算或校验逻辑。
assert validate_sequence(events)  # 执行本行的状态、计算或校验逻辑。
assert not validate_sequence([events[0], events[2]])  # 执行本行的状态、计算或校验逻辑。
assert not validate_sequence([events[0], StreamEvent("r-2", 2, "text.delta", "x")])  # 执行本行的状态、计算或校验逻辑。


## 3. 增量聚合：文本、工具参数与元数据各自维护缓冲区

把所有 event.data 拼成字符串会损坏工具 JSON、usage 或错误信息。消费者应按 kind 分类累积，再由终态决定是否对用户可见；真实服务还要限制单个流的内存和最大 event 大小。


In [ ]:
def consume(events):  # 执行本行的状态、计算或校验逻辑。
    state = {"text": "", "tool": "", "finish": None}  # 执行本行的状态、计算或校验逻辑。
    for event in events:  # 执行本行的状态、计算或校验逻辑。
        if event.kind == "text.delta":  # 执行本行的状态、计算或校验逻辑。
            state["text"] += event.data  # 执行本行的状态、计算或校验逻辑。
        elif event.kind == "tool.delta":  # 执行本行的状态、计算或校验逻辑。
            state["tool"] += event.data  # 执行本行的状态、计算或校验逻辑。
        elif event.kind == "finish":  # 执行本行的状态、计算或校验逻辑。
            state["finish"] = event.data  # 执行本行的状态、计算或校验逻辑。
    return state  # 执行本行的状态、计算或校验逻辑。
state = consume(events)  # 执行本行的状态、计算或校验逻辑。
assert state["text"] == "你好"  # 执行本行的状态、计算或校验逻辑。
assert state["tool"] == ""  # 执行本行的状态、计算或校验逻辑。
assert state["finish"] == "stop"  # 执行本行的状态、计算或校验逻辑。


## 4. 工具参数：只有完成的参数对象才能交给执行器

Agent 的工具参数可能被分成多个 chunk。渲染层可以显示进度，但执行边界必须等待 `tool.done` 或可验证的完整 JSON；否则截断参数会造成错误工具调用。


In [ ]:
tool_events = [StreamEvent("r-2", 1, "tool.delta", "{\"city\":"), StreamEvent("r-2", 2, "tool.delta", "\"上海\"}"), StreamEvent("r-2", 3, "tool.done", "weather")]  # 执行本行的状态、计算或校验逻辑。
tool_state = consume(tool_events)  # 执行本行的状态、计算或校验逻辑。
assert tool_state["tool"] == '{"city":"上海"}'  # 执行本行的状态、计算或校验逻辑。
assert tool_events[-1].kind == "tool.done"  # 执行本行的状态、计算或校验逻辑。
assert tool_state["finish"] is None  # 执行本行的状态、计算或校验逻辑。


## 5. 终态与 usage：只能出现一次且必须在所有 delta 之后

finish 是请求的提交点，usage 是计费/配额的最终记录。重复或提前的 finish 会导致客户端漏字、重复计费或错误结束；这里显式验证一个 terminal event 和末尾位置。


In [ ]:
def validate_terminal(events):  # 执行本行的状态、计算或校验逻辑。
    terminals = [index for index, event in enumerate(events) if event.kind in {"finish", "error"}]  # 执行本行的状态、计算或校验逻辑。
    return len(terminals) == 1 and terminals[0] == len(events) - 1  # 执行本行的状态、计算或校验逻辑。
assert validate_terminal(events)  # 执行本行的状态、计算或校验逻辑。
assert not validate_terminal(events + [StreamEvent("r-1", 4, "text.delta", "！")])  # 执行本行的状态、计算或校验逻辑。
assert not validate_terminal([events[0], StreamEvent("r-1", 2, "error", "timeout"), StreamEvent("r-1", 3, "finish", "stop")])  # 执行本行的状态、计算或校验逻辑。


## 6. 断连恢复：从已确认 offset 后重放，而不是重新拼全部

客户端应保存最后已验证 event id；服务端按 request、身份和保留窗口重放之后的事件。若历史已过期，必须明确返回不可恢复，让上层重新查询 request 状态，而不是无提示重复生成。


In [ ]:
def replay_after(events, last_sequence):  # 执行本行的状态、计算或校验逻辑。
    return [event for event in events if event.sequence > last_sequence]  # 执行本行的状态、计算或校验逻辑。
replayed = replay_after(events, 1)  # 执行本行的状态、计算或校验逻辑。
assert [event.sequence for event in replayed] == [2, 3]  # 执行本行的状态、计算或校验逻辑。
assert replay_after(events, 3) == []  # 执行本行的状态、计算或校验逻辑。
assert consume(replayed)["text"] == "好"  # 执行本行的状态、计算或校验逻辑。


## 7. 失败分支：错误事件也必须成为 terminal trace

上游超时、内容过滤、客户端取消和网络关闭不能都伪装成 stop。错误代码、可重试性和已生成 token 数应结构化记录；客户端据此决定显示部分文本、重试还是查询异步结果。


In [ ]:
error_events = [StreamEvent("r-3", 1, "text.delta", "部分"), StreamEvent("r-3", 2, "error", "upstream_timeout")]  # 执行本行的状态、计算或校验逻辑。
error_state = consume(error_events)  # 执行本行的状态、计算或校验逻辑。
assert validate_terminal(error_events)  # 执行本行的状态、计算或校验逻辑。
assert error_state["text"] == "部分"  # 执行本行的状态、计算或校验逻辑。
assert error_events[-1].data == "upstream_timeout"  # 执行本行的状态、计算或校验逻辑。


## 8. 指标与制品：TTFT、间隔、完成率与 replay 都要分开看

平均首 token 时间不能替代流完整性。至少记录 TTFT、token 间隔、终态完成率、重放成功率、取消率和每类 finish reason；审计制品需要绑定 API/schema/model 版本。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
metrics = {"ttft_ms": 120, "events": len(events), "terminal": validate_terminal(events), "replay_count": len(replayed)}  # 执行本行的状态、计算或校验逻辑。
digest = hashlib.sha256(json.dumps(metrics, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert metrics["terminal"] is True  # 执行本行的状态、计算或校验逻辑。
assert metrics["replay_count"] == 2  # 执行本行的状态、计算或校验逻辑。
assert len(digest) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时先说明不变量，再给出主路径和失败分支，最后说明指标、版本制品与生产替换点。不要把一个受控样例的通过误报成模型质量、可靠网络或端到端安全保证。
